In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv("heart_disease_uci.csv")

print(df.head())

In [ ]:
# Convert multi-class to binary
df['Heart_disease_detection'] = (df['Heart_disease_detection'] > 0).astype(int)

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
df['Resting_BP'] = df['Resting_BP'].fillna(df['Resting_BP'].mean())

In [ ]:
df['Cholesterol'] = df['Cholesterol'].fillna(df['Cholesterol'].mean())

In [ ]:
df['Fasting Blood Sugar'] = df['Fasting Blood Sugar'].fillna(df['Fasting Blood Sugar'].mean())

In [ ]:
df['Resting ECG Results'] = df['Resting ECG Results'].fillna(df['Resting ECG Results'].mode())[0]

In [ ]:
df['Maximum Heart Rate Achieved'] = df['Maximum Heart Rate Achieved'].fillna(df['Maximum Heart Rate Achieved'].mean())

In [ ]:
df['Exercise-Induced Angina'] = df['Exercise-Induced Angina'].fillna(df['Exercise-Induced Angina'].mean())

In [ ]:
df['oldpeak'] = df['oldpeak'].fillna(df['oldpeak'].mean())

In [ ]:
df['slope'] = df['slope'].fillna(df['slope'].mode()[0])

In [ ]:
df['Thalassemia'] = df['Thalassemia'].fillna(df['Thalassemia'].mode()[0])

In [ ]:
df.isnull().sum()

In [ ]:
df.drop('oldpeak.1',axis=1,inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
X = df.drop("Heart_disease_detection", axis=1)
y = df["Heart_disease_detection"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
svm = SVC(kernel='rbf', probability=True)
xgb = XGBClassifier(eval_metric='logloss')

estimators = [
    ('rf', rf),
    ('svm', svm),
    ('xgb', xgb)
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5
)

stack_model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = stack_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
y_prob = stack_model.predict_proba(X_test_scaled)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Stacked Model")
plt.legend()
plt.savefig("roc_curve.png")
plt.show()

print("ROC AUC Score:", roc_auc_score(y_test, y_prob))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix - Stacked Model")
plt.savefig("confusion_matrix.png")
plt.show()

In [ ]:

joblib.dump(scaler, "scaler.pkl")

In [ ]:
joblib.dump(stack_model, "heart_model.pkl")